DuckDB is used to perform analytical SQL queries directly on Pandas DataFrames, enabling efficient and scalable data analysis.

In [1]:
from google.colab import drive
import pandas as pd
import duckdb

# 1. Mount the Drive
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/IPL_PROJECT/IPL_EDA_Base.csv')
match_df=pd.read_csv('/content/drive/MyDrive/IPL_PROJECT/IPL_match_level.csv')


Mounted at /content/drive


In [ ]:
# Test query
duckdb.query("SELECT COUNT(*) FROM df").df()

,count_star()
0,278205


## **Top Run Scorers**

Question: Who are the highest run scorers?

In [ ]:
query="""
SELECT batter,
    sum(runs_batter) as total_runs
from df
Group by batter
order by total_runs desc
limit 10

"""
duckdb.query(query).df()

,batter,total_runs
0,V Kohli,8671.0
1,RG Sharma,7048.0
2,S Dhawan,6769.0
3,DA Warner,6567.0
4,SK Raina,5536.0
5,MS Dhoni,5439.0
6,KL Rahul,5235.0
7,AB de Villiers,5181.0
8,AM Rahane,5032.0
9,CH Gayle,4997.0


Insight:Top run scorers represent consistent performers who contribute significantly across seasons, forming the backbone of their teams

## **Strike Rate Analysis**

Question: Who are the most aggressive batters (with sufficient balls faced)?

In [41]:
query = """
SELECT batter,
       ROUND(
           SUM(runs_batter) * 100.0 /
           COUNT(CASE WHEN runs_extras = 0 THEN 1 END),
       2) AS strike_rate,
       COUNT(CASE WHEN runs_extras = 0 THEN 1 END) AS balls_faced
FROM df
GROUP BY batter
HAVING balls_faced > 500
ORDER BY strike_rate DESC
LIMIT 10
"""
duckdb.query(query).df()

,batter,strike_rate,balls_faced
0,PD Salt,182.07,580
1,AD Russell,178.19,1490
2,TM Head,175.23,654
3,SP Narine,174.00,1023
4,H Klaasen,172.90,856
5,N Pooran,171.63,1336
6,Abhishek Sharma,166.00,1094
7,LS Livingstone,163.71,642
8,V Sehwag,161.32,1691
9,GJ Maxwell,160.32,1759


Insight: Filtering batters based on minimum balls faced ensures that strike rate comparisons are meaningful. The results highlight players who combine aggressive scoring with sustained participation, making them valuable assets in high-scoring formats like T20 cricket.

## **TOP WICKET TAKERS**

Question: Who are the leading wicket takers?

In [3]:
df['wicket_kind'].value_counts()
df.head()

,match_id,season,date,innings,batting_team,bowling_team,over,ball,batter,bowler,...,toss_decision,player_of_match,winner,win_margin,is_four,is_six,is_boundary,is_dot,is_single,is_wicket
0,335982,2008,2008-04-18,1,KKR,RCB,0,1,SC Ganguly,P Kumar,...,field,BB McCullum,KKR,140 runs,0,0,0,0,1,0
1,335982,2008,2008-04-18,1,KKR,RCB,0,2,BB McCullum,P Kumar,...,field,BB McCullum,KKR,140 runs,0,0,0,1,0,0
2,335982,2008,2008-04-18,1,KKR,RCB,0,3,BB McCullum,P Kumar,...,field,BB McCullum,KKR,140 runs,0,0,0,0,1,0
3,335982,2008,2008-04-18,1,KKR,RCB,0,3,BB McCullum,P Kumar,...,field,BB McCullum,KKR,140 runs,0,0,0,1,0,0
4,335982,2008,2008-04-18,1,KKR,RCB,0,4,BB McCullum,P Kumar,...,field,BB McCullum,KKR,140 runs,0,0,0,1,0,0


In [36]:
query="""
SELECT bowler,
count(wicket_kind) as wickets
from df
WHERE wicket_kind IN ('caught','bowled','lbw','stumped','caught and bowled','hit wicket')
group by bowler
order by wickets desc
limit 10
"""
duckdb.query(query).df()


,bowler,wickets
0,YS Chahal,221
1,B Kumar,198
2,SP Narine,192
3,PP Chawla,192
4,R Ashwin,187
5,JJ Bumrah,186
6,DJ Bravo,183
7,A Mishra,174
8,RA Jadeja,170
9,SL Malinga,170


Insight: Accurate attribution of wickets is essential in cricket analytics, as including non-bowler dismissals like run-outs can significantly distort performance evaluation

## **ECONOMY RATE**

Question: Which bowlers are most economical?

In [18]:
query = """
WITH runs_conceded AS (
    SELECT bowler,
           SUM(runs_bowler) AS runs_conceded
    FROM df
    GROUP BY bowler
    HAVING SUM(runs_bowler) > 500
),

balls_bowled AS (
    SELECT bowler,
           COUNT(*) AS balls_bowled
    FROM df
    WHERE runs_extras = 0   -- assuming extras=0 means legal delivery
    GROUP BY bowler
    HAVING COUNT(*) > 500
)

SELECT r.bowler,
       ROUND(r.runs_conceded * 6.0 / b.balls_bowled, 2) AS economy_rate
FROM runs_conceded r
JOIN balls_bowled b
ON r.bowler = b.bowler
ORDER BY economy_rate
LIMIT 10
"""

duckdb.query(query).df()

,bowler,economy_rate
0,A Kumble,6.74
1,M Muralitharan,6.89
2,DL Vettori,6.90
3,SP Narine,6.91
4,J Botha,7.06
5,DW Steyn,7.08
6,R Sharma,7.10
7,Harbhajan Singh,7.18
8,Rashid Khan,7.22
9,R Ashwin,7.29


Insight: Accurate economy calculation requires distinguishing between legal and illegal deliveries, as including extras like wides can distort bowling workload and performance evaluation.

## **TOSS IMPACT ANALYSIS**

Question: Does winning the toss give an advantage?

In [19]:
query = """
SELECT toss_decision,
       COUNT(*) AS total_matches,
       SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END) AS wins_after_toss,
       ROUND(
           SUM(CASE WHEN toss_winner = winner THEN 1 ELSE 0 END) * 100.0
           / COUNT(*),
           2
       ) AS win_percentage
FROM match_df
GROUP BY toss_decision
"""
duckdb.query(query).df()

,toss_decision,total_matches,wins_after_toss,win_percentage
0,field,750,408.0,54.40
1,bat,396,183.0,46.21


Insight: The win percentage after winning the toss highlights that while toss decisions can influence match strategy, they do not guarantee victory, suggesting that in-game performance plays a more critical role.

## **Cumulative Runs**

In [40]:
query="""
WITH match_runs AS (
    SELECT batter, match_id, SUM(runs_batter) AS runs
    FROM df
    GROUP BY batter, match_id
)

SELECT batter,
       match_id,
       SUM(runs) OVER (
           PARTITION BY batter
           ORDER BY match_id
       ) AS cumulative_runs
FROM match_runs
"""
duckdb.query(query).df()

,batter,match_id,cumulative_runs
0,AA Bilakhia,392196,12.0
1,AA Bilakhia,392201,34.0
2,AA Bilakhia,392205,35.0
3,AA Bilakhia,392236,51.0
4,AA Bilakhia,392237,61.0
...,...,...,...
17633,Y Nagar,548364,285.0
17634,Yash Dayal,1304081,0.0
17635,Yash Dayal,1473471,0.0
17636,Yash Dayal,1473503,3.0


## **RUNNING AVERAGE**

In [38]:
query = """
SELECT batter,
       match_id,
       ROUND(AVG(runs_batter) OVER (
           PARTITION BY batter
           ORDER BY match_id),2
       ) AS running_avg
FROM df
"""
duckdb.query(query).df()

,batter,match_id,running_avg
0,A Tomar,1304112,0.50
1,A Tomar,1304112,0.50
2,A Tomar,1304112,0.50
3,A Tomar,1304112,0.50
4,A Tomar,1304112,0.50
...,...,...,...
278200,Yash Dayal,1473503,0.38
278201,Yash Dayal,1473503,0.38
278202,Yash Dayal,1473503,0.38
278203,Yash Dayal,1473503,0.38


## **JOIN-Based Analysis**

Question: How does toss decision affect total score?

In [31]:
query = """
SELECT m.toss_decision,
       ROUND(AVG(t.total_runs), 2) AS avg_total
FROM match_df m
JOIN (
    SELECT match_id, innings, SUM(runs_total) AS total_runs
    FROM df
    WHERE innings = 1
    GROUP BY match_id, innings
) t
ON m.match_id = t.match_id
GROUP BY m.toss_decision
"""
duckdb.query(query).df()

,toss_decision,avg_total
0,bat,161.76
1,field,170.20


Insight: Teams choosing to bat or field first show differences in average first innings totals, reflecting strategic decisions based on pitch conditions and match dynamics

## **Player Performance In Winning Matches**

Quesstion: Which players have the best performance in winning matches?

In [32]:
query = """
SELECT d.batter,
       SUM(d.runs_batter) AS runs_in_wins
FROM df d
JOIN match_df m
ON d.match_id = m.match_id
WHERE d.batting_team = m.winner
GROUP BY d.batter
ORDER BY runs_in_wins DESC
LIMIT 10
"""
duckdb.query(query).df()

,batter,runs_in_wins
0,V Kohli,4789.0
1,RG Sharma,4276.0
2,S Dhawan,3945.0
3,DA Warner,3710.0
4,SK Raina,3559.0
5,CH Gayle,3116.0
6,MS Dhoni,3050.0
7,AB de Villiers,2967.0
8,F du Plessis,2855.0
9,G Gambhir,2839.0


Insight: This analysis identifies players who contribute heavily in winning matches, highlighting true match-winners rather than just high scorers

Question: Which teams have won the most matches?

In [26]:
query = """
SELECT winner,
       COUNT(*) AS total_wins
FROM match_df
GROUP BY winner
ORDER BY total_wins DESC
LIMIT 10
"""
duckdb.query(query).df()

,winner,total_wins
0,MI,151
1,CSK,142
2,KKR,135
3,RCB,132
4,SRH,122
5,PBKS,119
6,DC,118
7,RR,114
8,GT,37
9,LSG,30


## **Death Overs Performance in Wins**

In [35]:
query = """
SELECT d.batter,
       SUM(d.runs_batter) AS death_runs_in_wins
FROM df d
WHERE over >= 15
  AND batting_team = winner
GROUP BY batter
ORDER BY death_runs_in_wins DESC
LIMIT 10
"""
duckdb.query(query).df()

,batter,death_runs_in_wins
0,MS Dhoni,2007.0
1,AB de Villiers,1334.0
2,KA Pollard,1267.0
3,KD Karthik,1034.0
4,RG Sharma,1027.0
5,HH Pandya,909.0
6,DA Miller,885.0
7,RA Jadeja,844.0
8,V Kohli,805.0
9,AT Rayudu,778.0


Insight: Players who score heavily in death overs of winning matches demonstrate the ability to finish games under pressure, making them valuable match-winner